# AlignLab — Experiment 02: Base vs. QLoRA Evaluation

Evaluate **Qwen/Qwen3-1.7B** and the fine-tuned adapter from `hamedready/alignlab-qwen3-1.7b-qlora` using identical held-out instructions.

**Kaggle automated workflow:** GitHub Actions downloads the private adapter using its `HF_TOKEN`, stages it in the private Kaggle dataset `hamedhani/alignlab-qlora-adapter`, and attaches that dataset to the evaluation notebook. The notebook loads the **local dataset**; it never needs a Kaggle HF secret.

**Colab:** Supply `HF_TOKEN` in Colab Secrets to load the private adapter from the Hub, or provide a local adapter directory.

The benchmark scores **full-sequence reference-token NLL and perplexity**, not instruction accuracy. It uses 40 examples from `yahma/alpaca-cleaned` after the original 120 training/validation examples (assuming Experiment 01 used the same shuffle and split).


## 0. Restrict GPU visibility before CUDA imports

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

## 1. Install dependencies

Run once, then restart runtime if your environment requests it; rerun the notebook from the top.

In [ ]:
%pip install -q "transformers>=4.51,<5" "peft>=0.15,<0.19" \
    "accelerate>=1.4,<2" "datasets>=3.5,<5" \
    "bitsandbytes>=0.45" "huggingface-hub>=0.30,<1" \
    "sentencepiece"

## 2. Settings and environment

In [ ]:
import json
import math
import time
from datetime import UTC, datetime
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable an NVIDIA GPU runtime."
assert torch.cuda.device_count() == 1, "Restart runtime: exactly one GPU must be visible."

MODEL_ID = "Qwen/Qwen3-1.7B"
ADAPTER_REPO = "hamedready/alignlab-qwen3-1.7b-qlora"
SEED = 42
MAX_LENGTH = 512
EVAL_EXAMPLES = 40
GEN_EXAMPLES = 5
MAX_NEW_TOKENS = 128

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/alignlab-evaluation")
elif Path("/content").exists():
    OUTPUT_DIR = Path("/content/alignlab-evaluation")
else:
    OUTPUT_DIR = Path("./alignlab-evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT_DIR)

## 3. Load model and tokenizer

Use the same NF4 quantization for both evaluations. The adapter is applied to the same loaded base model to minimize memory usage.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16,
)
model.eval()
print("Loaded base:", MODEL_ID, "device:", model.device)

## 4. Load held-out examples

Experiment 01 shuffled `yahma/alpaca-cleaned` with seed 42 and selected the first 120 items (100 train + 20 validation). We select indices **120–159** from the same shuffled dataset to avoid those examples. If you changed Experiment 01 dataset selection, revise this split accordingly.

In [ ]:
from datasets import load_dataset

raw = load_dataset("yahma/alpaca-cleaned", split="train")
shuffled = raw.shuffle(seed=SEED)
assert len(shuffled) >= 120 + EVAL_EXAMPLES
heldout = shuffled.select(range(120, 120 + EVAL_EXAMPLES))


def user_text(example):
    instruction = example["instruction"].strip()
    context = example["input"].strip()
    return instruction if not context else f"{instruction}\n\nInput:\n{context}"


def full_chat(example):
    return tokenizer.apply_chat_template(
        [
            {"role": "user", "content": user_text(example)},
            {"role": "assistant", "content": example["output"].strip()},
        ],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )


print("Held-out examples:", len(heldout))
print("Example:", user_text(heldout[0])[:200])

## 5. Evaluation helpers

NLL measures how well the model predicts reference tokens. It is not a direct measure of response quality. Both models use identical tokenization and truncation. We score full user+assistant chat tokens (including prompt tokens); use response-only masking for a stricter assistant-only evaluation later.

In [ ]:
def score_nll(active_model, example):
    text = full_chat(example)
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )
    device = active_model.get_input_embeddings().weight.device
    enc = {key: val.to(device) for key, val in enc.items()}
    length = int(enc["input_ids"].shape[1])
    if length < 2:
        return None
    with torch.inference_mode():
        outputs = active_model(**enc, labels=enc["input_ids"])
    return {"nll_sum": float(outputs.loss.item()) * (length - 1), "tokens": length - 1}


def generate(active_model, example):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text(example)}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    enc = tokenizer(prompt, return_tensors="pt")
    device = active_model.get_input_embeddings().weight.device
    enc = {key: value.to(device) for key, value in enc.items()}
    active_model.eval()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        ids = active_model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    answer_ids = ids[0, enc["input_ids"].shape[1] :]
    return {
        "response": tokenizer.decode(answer_ids, skip_special_tokens=True),
        "latency_seconds": elapsed,
        "generated_tokens": int(answer_ids.shape[0]),
    }


def evaluate_model(active_model, examples, n_generations):
    scores = []
    generations = []
    for i, example in enumerate(examples):
        score = score_nll(active_model, example)
        if score is not None:
            scores.append(score)
        if i < n_generations:
            generations.append(generate(active_model, example))
        if (i + 1) % 10 == 0:
            print(f"Evaluated {i + 1}/{len(examples)}")
    nll_sum = sum(s["nll_sum"] for s in scores)
    token_count = sum(s["tokens"] for s in scores)
    mean_nll = nll_sum / token_count
    return {
        "mean_token_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "scored_tokens": token_count,
        "examples_scored": len(scores),
        "mean_generation_latency_seconds": (
            sum(g["latency_seconds"] for g in generations) / len(generations)
            if generations
            else None
        ),
        "generations": generations,
    }

## 6. Evaluate base model

In [ ]:
base_results = evaluate_model(model, heldout, GEN_EXAMPLES)
print("Base mean token NLL:", round(base_results["mean_token_nll"], 4))
print("Base perplexity:", round(base_results["perplexity"], 4))

## 7. Load the fine-tuned adapter

On Kaggle, this notebook **requires** the private Kaggle dataset `hamedhani/alignlab-qlora-adapter` to be attached by `scripts/prepare_kaggle_evaluation.py`. If the dataset is absent, the notebook stops immediately with a useful error instead of attempting an unauthenticated Hub download.

On Colab, add `HF_TOKEN` to Secrets (not a code cell) or set `ADAPTER_LOCAL_PATH`.


In [ ]:
from peft import PeftModel

# Optional override for Colab or a manually attached local adapter.
ADAPTER_LOCAL_PATH = None
KAGGLE_ADAPTER_DATASET = "alignlab-qlora-adapter"


def get_optional_hf_token():
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token
    if Path("/content").exists() and not Path("/kaggle/working").exists():
        try:
            from google.colab import userdata

            return (userdata.get("HF_TOKEN") or "").strip() or None
        except (ImportError, KeyError, RuntimeError):
            return None
    return None


def find_local_adapter():
    if ADAPTER_LOCAL_PATH:
        folder = Path(ADAPTER_LOCAL_PATH)
        if not (folder / "adapter_config.json").is_file():
            raise FileNotFoundError(folder / "adapter_config.json")
        return folder

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        dataset_root = kaggle_root / KAGGLE_ADAPTER_DATASET
        if not dataset_root.exists():
            raise FileNotFoundError(
                f"Missing attached Kaggle dataset: {dataset_root}. "
                "Check kernel-metadata.json dataset_sources and the "
                "GitHub Actions adapter-staging step."
            )
        matches = sorted(dataset_root.rglob("adapter_config.json"))
        if len(matches) != 1:
            raise RuntimeError(
                f"Expected one adapter_config.json under {dataset_root}; found {len(matches)}."
            )
        return matches[0].parent
    return None


local_adapter = find_local_adapter()
if local_adapter is not None:
    if not any(
        (local_adapter / filename).is_file()
        for filename in ("adapter_model.safetensors", "adapter_model.bin")
    ):
        raise FileNotFoundError(f"Adapter weights missing from {local_adapter}")
    adapter_source = str(local_adapter)
    hf_token = None
else:
    adapter_source = ADAPTER_REPO
    hf_token = get_optional_hf_token()
    if not hf_token:
        raise RuntimeError(
            "Private Hugging Face adapter requires HF_TOKEN in Colab Secrets, "
            "or set ADAPTER_LOCAL_PATH to a downloaded adapter directory."
        )

print("Adapter source:", adapter_source)
model = PeftModel.from_pretrained(
    model,
    adapter_source,
    token=hf_token,
    is_trainable=False,
)
model.eval()
print("LoRA adapter loaded successfully.")

## 8. Evaluate the fine-tuned model

In [ ]:
sft_results = evaluate_model(model, heldout, GEN_EXAMPLES)
print("SFT mean token NLL:", round(sft_results["mean_token_nll"], 4))
print("SFT perplexity:", round(sft_results["perplexity"], 4))

## 9. Compare and save results

Lower NLL/perplexity means better prediction of these held-out reference tokens. Generation latency is only a rough indicator, especially when measured sequentially without repeated warmups.

In [ ]:
import peft
import transformers

comparisons = []
for i in range(GEN_EXAMPLES):
    comparisons.append(
        {
            "prompt": user_text(heldout[i]),
            "reference": heldout[i]["output"],
            "base": base_results["generations"][i],
            "sft": sft_results["generations"][i],
        }
    )

report = {
    "experiment": "02_model_evaluation",
    "timestamp_utc": datetime.now(UTC).isoformat(),
    "base_model": MODEL_ID,
    "adapter": ADAPTER_REPO,
    "adapter_source": adapter_source,
    "dataset": "yahma/alpaca-cleaned",
    "evaluation_split": "shuffle(seed=42)[120:160]",
    "evaluation_examples": EVAL_EXAMPLES,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "peft_version": peft.__version__,
    "base": base_results,
    "sft": sft_results,
    "comparisons": comparisons,
}
report_path = OUTPUT_DIR / "model_comparison.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print("Saved:", report_path)
print("Base NLL:", round(base_results["mean_token_nll"], 4))
print("SFT NLL:", round(sft_results["mean_token_nll"], 4))
print(
    "Delta (SFT - Base):", round(sft_results["mean_token_nll"] - base_results["mean_token_nll"], 4)
)
for i, item in enumerate(comparisons[:3], 1):
    print(f"\nExample {i}: {item['prompt'][:120]}")
    print("Base:", item["base"]["response"][:250])
    print("SFT:", item["sft"]["response"][:250])

## 10. Export

`model_comparison.json` is saved under `/kaggle/working/alignlab-evaluation` on Kaggle or `/content/alignlab-evaluation` on Colab. The evaluation GitHub Actions workflow downloads it, validates the JSON, and commits a unique run directory to `results/evaluation/`.


In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Evaluation archive:", archive)

## Interpretation and next experiment

- This comparison measures held-out **full-sequence token NLL**, not task accuracy. A lower value is evidence of better fit to the reference distribution, not necessarily better user-facing answers.
- Inspect the side-by-side examples manually. For rigorous instruction following, add response-only NLL and a separately labeled evaluation set with rubric-based scoring.
- The training experiment used only 100 examples and 30 steps; improvements may be small or inconsistent.
- **Experiment 03:** build a task-specific code-generation SFT dataset, exclude HumanEval from training, and evaluate generated code in an isolated test runner.